# Rung 4 — Intervals, and the place where you are allowed to stop

**Question.** How uncertain is the fit?

This is the rung the whole plan singles out. If the task is *fit this model and say how
confident you are*, the answer is here and climbing further buys nothing. The reason to
keep going is not better error bars — it is wanting a **joint** distribution to push
through a decision, which is rung 5.

Three ways to get an interval, in increasing cost and honesty:

| method | cost | assumes |
|---|---|---|
| Laplace | one Hessian | the likelihood is a Gaussian bump in log-parameters |
| profile | one optimisation per grid point | nothing about shape; one parameter at a time |
| parametric bootstrap | one full refit per replicate | that the fitted model is right |

Where they agree, that is evidence. Where they disagree, the cheap one is wrong.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from inferladder import default_sir_dataset, default_sir_model
from inferladder.diagnostics import observed_fisher, profile_interval, profile_likelihood
from inferladder.inference import (
    fit,
    laplace_intervals,
    negative_log_likelihood,
    parametric_bootstrap,
)

dataset = default_sir_dataset()
model = default_sir_model()
truth = dataset.truth
objective = negative_log_likelihood(model, dataset.y, dataset.t)
mle = fit(objective, dataset.theta_true * 1.2, model.param_names)
mle.as_dict

## Laplace: invert the curvature

One Hessian at the optimum, and every interval is free after that.

In [ ]:
hessian = observed_fisher(objective, mle.theta)
laplace = laplace_intervals(mle.theta, hessian, model.param_names)

print(f"{'param':8} {'estimate':>10} {'95% interval':>26} {'truth':>9}  covers")
for name in model.param_names:
    low, high = laplace[name]
    covers = "yes" if low <= truth[name] <= high else "NO"
    print(
        f"{name:8} {mle[name]:10.4f} {f'[{low:.4f}, {high:.4f}]':>26} {truth[name]:9.4f}  {covers}"
    )

Three of the four intervals cover the truth. The dispersion `k` does not — and that is
not bad luck, it is the Laplace approximation being asked to describe a direction where
the likelihood is not remotely quadratic. `k` is the least constrained parameter in the
model, which is exactly the case where the cheap method fails, and it fails *quietly*:
nothing in the output above flags it.

## Profile: no assumption about shape

In [ ]:
profiles, profile_ci = {}, {}
for name in ("beta", "gamma", "rho"):
    index = model.param_names.index(name)
    centre = mle.theta[index]
    grid = np.linspace(centre * 0.80, centre * 1.25, 15)
    prof = profile_likelihood(objective, mle.theta, index, grid, model.param_names)
    profiles[name] = prof
    profile_ci[name] = profile_interval(prof)

for name, (low, high) in profile_ci.items():
    print(
        f"{name:6} profile 95% [{low:.4f}, {high:.4f}]   Laplace "
        f"[{laplace[name][0]:.4f}, {laplace[name][1]:.4f}]"
    )

## Bootstrap: simulate, refit, look at the spread

The most expensive and the least assuming about the *estimator*: it re-runs the whole
pipeline, so an optimiser that only sometimes converges shows up here and nowhere else.

The replicate count below is kept modest so the notebook stays runnable in CI. It is the
one number here worth raising for a result you intend to publish.

In [ ]:
boot = parametric_bootstrap(model, mle.theta, dataset.t, n_replicates=50, seed=4)
print(f"replicates used: {len(boot.samples)}   failed refits: {boot.n_failed}")

for name in model.param_names:
    low, high = boot.interval(name)
    print(f"{name:6} bootstrap 95% [{low:.4f}, {high:.4f}]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, name in zip(axes, ("beta", "gamma", "rho"), strict=True):
    column = boot.samples[:, model.param_names.index(name)]
    ax.hist(column, bins=16, color="#33628f", alpha=0.75)
    ax.axvline(mle[name], color="#c1502e", lw=2, label="fit")
    ax.axvline(truth[name], color="#3c3c3c", ls="--", lw=2, label="truth")
    ax.set(xlabel=name, ylabel="replicates")
    ax.legend(fontsize=8)
fig.suptitle("Bootstrap sampling distributions")
fig.tight_layout()

## The three side by side

Agreement between three methods that assume different things is worth more than any one
of them alone.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
for ax, name in zip(axes, ("beta", "gamma", "rho"), strict=True):
    entries = [
        ("Laplace", laplace[name]),
        ("profile", profile_ci[name]),
        ("bootstrap", boot.interval(name)),
    ]
    for row, (_label, (low, high)) in enumerate(entries):
        ax.plot([low, high], [row, row], "-", lw=3, color="#33628f")
        ax.plot([mle[name]], [row], "o", color="#c1502e", ms=6)
    ax.axvline(truth[name], ls="--", color="#3c3c3c", lw=1.5)
    ax.set(yticks=range(3), yticklabels=[e[0] for e in entries], xlabel=name, ylim=(-0.6, 2.6))
    ax.grid(alpha=0.3, axis="x")
fig.suptitle("95% intervals, three ways (dashed line = truth)")
fig.tight_layout()

## A quantity nobody parameterised

`R0 = beta / gamma` is not a parameter of the model, and the intervals above say nothing
about it directly — `beta` and `gamma` are strongly correlated, so combining their
separate intervals would be wrong. The bootstrap gives it for free, correlations included,
with no delta-method algebra.

In [ ]:
r0 = boot.derived(lambda theta: theta[0] / theta[1])
low, high = np.percentile(r0, [2.5, 97.5])

print(f"R0 point estimate : {mle['beta'] / mle['gamma']:.3f}")
print(f"R0 bootstrap 95%  : [{low:.3f}, {high:.3f}]")
print(f"R0 truth          : {truth['beta'] / truth['gamma']:.3f}")
print(f"P(R0 > 2) = {(r0 > 2).mean():.2f}    P(R0 > 3) = {(r0 > 3).mean():.2f}")

That last line is the first genuine *decision* quantity in this notebook: not an
estimate, but a probability attached to a threshold that matters.

## When to stop here

**This is the stopping point for most fitting work.** You have a point estimate, you have
checked it is the optimum (rung 2), you know which parameters the data determine (rung 3),
and you have intervals from three methods that agree. For "fit this model and report
uncertainty", nothing above this rung is needed.

**Three things would push you higher.**

- *The cheap approximation failed somewhere.* `k` above is the warning. Laplace missed it,
  and only a method making no quadratic assumption caught that.
- *You want to use prior information.* Nothing here can express "serial interval studies
  put `1/gamma` near 7 days". The likelihood has no channel for it.
- *You want a joint distribution, not a set of intervals.* Every interval here is
  one-dimensional. Propagating uncertainty through a decision — expected hospitalisations
  under an intervention, with correlations intact — needs the joint posterior.

That is rung 5, and it is the first rung where the reason to climb is a decision rather
than a diagnostic.